# AI Stock Agent — Demo Notebook

Demonstrates end-to-end invocation of the deployed AWS AgentCore endpoint:
1. Cognito authentication
2. Streaming queries against the agent
3. Langfuse observability traces


## 0. Prerequisites

```bash
pip install boto3 requests
```

Fill in the configuration cell below with values from `terraform output`.

In [1]:
#!pip3 show boto3

In [2]:
#!pip3 install boto3 requests --break-system-packages

In [3]:
#!pip3 install --upgrade boto3 botocore --break-system-packages

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from repo root (one level up from notebooks/)
load_dotenv(Path('..') / '.env')

AWS_REGION           = os.environ['AWS_REGION']
COGNITO_USER_POOL_ID = os.environ['COGNITO_USER_POOL_ID']
COGNITO_CLIENT_ID    = os.environ['COGNITO_APP_CLIENT_ID']
AWS_PROFILE          = os.environ['AWS_PROFILE']  # setup your profile with aws cli: aws configure --profile <profile_name>

# Cognito test user credentials
COGNITO_USERNAME = os.environ.get('COGNITO_USERNAME', 'efloresd1794@gmail.com')
COGNITO_PASSWORD = os.environ.get('COGNITO_PASSWORD', 'Demo1234!$')

# AgentCore endpoint — set AGENT_ENDPOINT in .env after terraform apply
AGENT_ENDPOINT = os.environ.get('AGENT_ENDPOINT', 'arn:aws:bedrock-agentcore:us-east-1:837271987159:runtime/amzn_stock_agent_runtime-lpQzbn56k2')
QUERY_PATH     = '/query'

print(f'Region          : {AWS_REGION}')
print(f'User Pool ID    : {COGNITO_USER_POOL_ID}')
print(f'App Client ID   : {COGNITO_CLIENT_ID}')
print(f'Username        : {COGNITO_USERNAME}')
print(f'Agent Endpoint  : {AGENT_ENDPOINT or "(not set yet)"}')


Region          : us-east-1
User Pool ID    : us-east-1_xKlWv8xHC
App Client ID   : 25ddhr2df4s7cb75h3qg1iuina
Username        : efloresd1794@gmail.com
Agent Endpoint  : arn:aws:bedrock-agentcore:us-east-1:837271987159:runtime/amzn_stock_agent_runtime-lpQzbn56k2


In [6]:
import boto3

session = boto3.Session(profile_name=AWS_PROFILE)


cognito = session.client(
    'cognito-idp',
    region_name=AWS_REGION
)
#cognito = boto3.client('cognito-idp', region_name=AWS_REGION)

# ── Create user ───────────────────────────────────────────────────────────────
try:
    cognito.admin_create_user(
        UserPoolId=COGNITO_USER_POOL_ID,
        Username=COGNITO_USERNAME,
        TemporaryPassword=COGNITO_PASSWORD,
        MessageAction='SUPPRESS',  # don't send welcome email
    )
    print(f'User created: {COGNITO_USERNAME}')
except cognito.exceptions.UsernameExistsException:
    print(f'User already exists: {COGNITO_USERNAME} — skipping')

# ── Set permanent password (bypasses FORCE_CHANGE_PASSWORD state) ─────────────
cognito.admin_set_user_password(
    UserPoolId=COGNITO_USER_POOL_ID,
    Username=COGNITO_USERNAME,
    Password=COGNITO_PASSWORD,
    Permanent=True,
)
print(f'Permanent password set for: {COGNITO_USERNAME}')


User already exists: efloresd1794@gmail.com — skipping
Permanent password set for: efloresd1794@gmail.com


## 1. Cognito Authentication

Authenticate the demo user and obtain a JWT access token.

In [2]:
import boto3

session = boto3.Session(profile_name=AWS_PROFILE)

cognito = session.client(
    'cognito-idp',
    region_name=AWS_REGION
)

auth_response = cognito.initiate_auth(
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': COGNITO_USERNAME, 'PASSWORD': COGNITO_PASSWORD},
    ClientId=COGNITO_CLIENT_ID,
)

JWT_TOKEN     = auth_response['AuthenticationResult']['AccessToken']
REFRESH_TOKEN = auth_response['AuthenticationResult']['RefreshToken']

print('Authentication successful')
print(f"Access token (first 60 chars): {JWT_TOKEN[:60]}...")
print(f"Expires in: {auth_response['AuthenticationResult']['ExpiresIn']}s")

Authentication successful
Access token (first 60 chars): eyJraWQiOiJqNVBONE9YYjFuYW1yaWllb3poSkpDTWIrQyt3WkoyZGFCRHlK...
Expires in: 3600s


## 2. Streaming Query Helper

In [9]:
import boto3
import json
import uuid

session = boto3.Session(profile_name=AWS_PROFILE)

agentcore = session.client(
    "bedrock-agentcore",
    region_name=AWS_REGION
)

def query_agent(question: str, token: str = None) -> str:
    """Invoke AgentCore runtime and stream the response."""
    _token = token or JWT_TOKEN
    session_id = str(uuid.uuid4())  # must be 33+ chars; uuid4 = 36 chars

    payload = json.dumps({'query': question, 'token': _token})

    print(f'Query: {question}')
    print('─' * 60)

    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ENDPOINT,
        runtimeSessionId=session_id,
        payload=payload,
    )

    # Stream response body
    full = ''
    streaming_body = response.get('response') or response.get('body')
    for chunk in streaming_body.iter_chunks():
        text = chunk.decode('utf-8') if isinstance(chunk, bytes) else chunk
        full += text
        print(text, end='', flush=True)

    print('\n' + '─' * 60)
    return full

print('Helper ready.')


Helper ready.


## 3. Query 1 — Real-time Stock Price

> *What is the stock price for Amazon right now?*

In [10]:
r1 = query_agent('What is the stock price for Amazon right now?')

Query: What is the stock price for Amazon right now?
────────────────────────────────────────────────────────────
I'll retrieve the current real-time stock price for Amazon right away.
[tool_call] retrieve_realtime_stock_price
I apologize, but there seems to be a technical issue retrieving the real-time stock price at the moment. This can happen due to various reasons such as market closed, temporary system constraints, or connection issues. 

For the most up-to-date stock price, I recommend:
1. Checking a financial website like Yahoo Finance or Google Finance
2. Consulting your brokerage platform
3. Checking the NASDAQ or NYSE website for the most current AMZN stock price

Would you like me to help you find the most recent stock information through another method?
────────────────────────────────────────────────────────────


## 4. Query 2 — Historical Prices Q4

> *What were the stock prices for Amazon in Q4 last year?*

In [11]:
r2 = query_agent('What were the stock prices for Amazon in Q4 last year?')

Query: What were the stock prices for Amazon in Q4 last year?
────────────────────────────────────────────────────────────
I'll help you retrieve the historical stock prices for Amazon in Q4 of last year. I'll use the retrieve_historical_stock_price function to fetch this data.

First, I'll define the date range for Q4 of last year, which is from October 1st to December 31st, 2023.
[tool_call] retrieve_historical_stock_price
I apologize, but it seems there might be an issue retrieving the historical data. Let me try the current year's Q4 (2023) to verify the data availability.
[tool_call] retrieve_historical_stock_price
I apologize, but I'm experiencing technical difficulties retrieving the historical stock prices. To provide you with the most accurate and up-to-date information, I recommend:

1. Checking a financial website like Yahoo Finance or Google Finance for precise historical stock prices.
2. Verifying the stock ticker (though I'm certain AMZN is correct).

Let me get the curre

## 5. Query 3 — Performance vs Analyst Predictions

> *Compare Amazon's recent stock performance to what analysts predicted in their reports.*

In [6]:
r3 = query_agent("Compare Amazon's recent stock performance to what analysts predicted in their reports")

Query: Compare Amazon's recent stock performance to what analysts predicted in their reports
────────────────────────────────────────────────────────────
I'll help you compare Amazon's recent stock performance with analyst predictions. I'll break this down into a few steps:

1. First, let's check the current stock price:
[tool_call] retrieve_realtime_stock_price
I'll now search the financial reports for analyst predictions and performance insights:
[tool_call] retrieve_from_knowledge_base
Let me break down the key insights:

Stock Performance:
1. According to the Stock Price Performance Graph in the 2024 Annual Report, Amazon's stock has shown strong growth:
   - From 2019 to 2024, the stock value increased from $100 to $237
   - This outperformed both the S&P 500 Index and the S&P 500 Consumer Discretionary Distribution & Retail Index

Analyst Predictions and Financial Performance:
1. Q1 2025 Guidance (from February 6, 2025):
   - Net sales expected between $151.0 billion and $155.5 b

## 6. Query 4 — Current Price + AI Business

> *I'm researching AMZN — give me the current price and any relevant information about their AI business.*

In [7]:
r4 = query_agent("I'm researching AMZN, give me the current price and any relevant information about their AI business")

Query: I'm researching AMZN, give me the current price and any relevant information about their AI business
────────────────────────────────────────────────────────────
I'll help you with that by first retrieving the current stock price and then searching for information about Amazon's AI business in their recent financial reports.

First, let's get the current stock price:
[tool_call] retrieve_realtime_stock_price
I apologize, but there seems to be an issue retrieving the real-time stock price. Let me search for information about Amazon's AI business in their recent financial reports:
[tool_call] retrieve_from_knowledge_base
Based on the financial reports, here are the key insights about Amazon's AI business:

1. AI Strategy Overview:
- Amazon is heavily investing in AI, with over 1,000 generative AI applications being developed across the company.
- They aim to reinvent customer experiences in various sectors including shopping, coding, personal assistants, streaming, advertising, he

## 7. Query 5 — Office Space (RAG)

> *What is the total amount of office space Amazon owned in North America in 2024?*

In [8]:
r5 = query_agent('What is the total amount of office space Amazon owned in North America in 2024?')

Query: What is the total amount of office space Amazon owned in North America in 2024?
────────────────────────────────────────────────────────────
Let me search the Amazon financial reports to find information about their office space.
[tool_call] retrieve_from_knowledge_base
Based on the 2024 Annual Report, Amazon owned 9,104 thousand square feet (9.104 million square feet) of office space in North America as of December 31, 2024. 

To provide more context, this is part of a larger real estate portfolio that includes:
- Total owned office space in North America: 9,104,000 sq ft
- Total leased office space in North America: 29,551,000 sq ft
- Total owned square footage across all property types in North America: 20,741,000 sq ft

This information comes directly from the "Properties" section of Amazon's 2024 Annual Report, specifically Page 29, which provides a detailed breakdown of Amazon's owned and leased facilities by type and location.
─────────────────────────────────────────────

## 8. Langfuse Observability Traces

Traces are automatically captured after every query. The cell below fetches them via the Langfuse API.

In [28]:
import os
import requests as req
from IPython.display import display, JSON

# Keys already loaded from .env via dotenv in cell 0
LANGFUSE_SECRET_KEY = os.environ['LANGFUSE_SECRET_KEY']
LANGFUSE_PUBLIC_KEY = os.environ['LANGFUSE_PUBLIC_KEY']
# support both LANGFUSE_HOST and LANGFUSE_BASE_URL
LANGFUSE_HOST = os.environ.get('LANGFUSE_HOST') or os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')

resp = req.get(
    f'{LANGFUSE_HOST}/api/public/traces',
    auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
    params={'limit': 5},
    timeout=15,
)
resp.raise_for_status()
traces = resp.json()

print(f"Latest {len(traces.get('data', []))} traces:")
for t in traces.get('data', []):
    print(f"  id={t['id']}  name={t.get('name')}  userId={t.get('userId')}  createdAt={t.get('createdAt')}")
display(JSON(traces))


Latest 5 traces:
  id=6a5a8b5a-87a7-4f9f-bbc1-6a4a5059aa72  name=LangGraph  userId=a4a814a8-7071-70aa-83b5-dc49836a0c39  createdAt=2026-03-10T22:13:14.000Z
  id=e987414f-d392-4f25-b529-7ec66bca47b0  name=LangGraph  userId=a4a814a8-7071-70aa-83b5-dc49836a0c39  createdAt=2026-03-10T22:12:42.000Z
  id=f2461f8a-766f-4691-8d44-eb27e5821796  name=LangGraph  userId=a4a814a8-7071-70aa-83b5-dc49836a0c39  createdAt=2026-03-10T22:12:16.000Z
  id=63a2aeca-52d5-46fb-a7f1-c7d1f8e36713  name=LangGraph  userId=a4a814a8-7071-70aa-83b5-dc49836a0c39  createdAt=2026-03-10T22:11:49.000Z
  id=e0f88971-5b50-45eb-93e8-fba4bf415877  name=LangGraph  userId=a4a814a8-7071-70aa-83b5-dc49836a0c39  createdAt=2026-03-10T22:11:29.000Z


<IPython.core.display.JSON object>

In [29]:
# Fetch detail for the most recent trace
if traces.get('data'):
    latest_id = traces['data'][0]['id']
    detail = req.get(
        f"{LANGFUSE_HOST}/api/public/traces/{latest_id}",
        auth=(LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY),
        timeout=15,
    ).json()
    print(f"Trace: {latest_id}")
    print(f"  Input : {str(detail.get('input', ''))[:200]}")
    print(f"  Output: {str(detail.get('output', ''))[:200]}")
    print(f"\nSpan breakdown:")
    for obs in detail.get('observations', []):
        print(f"  [{obs.get('type')}] {obs.get('name')}  latency={obs.get('latency')}ms")

Trace: 6a5a8b5a-87a7-4f9f-bbc1-6a4a5059aa72
  Input : {'messages': [{'content': 'What is the total amount of office space Amazon owned in North America in 2024?', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'd3a
  Output: {'messages': [{'content': 'What is the total amount of office space Amazon owned in North America in 2024?', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'd3a

Span breakdown:
  [GENERATION] ChatBedrock  latency=3.65ms
  [SPAN] retrieve_from_knowledge_base  latency=0.122ms
  [SPAN] should_continue  latency=0.002ms
  [SPAN] _write  latency=0.001ms
  [SPAN] call_model  latency=3.656ms
  [SPAN] should_continue  latency=0.002ms
  [SPAN] agent  latency=3.662ms
  [SPAN] LangGraph  latency=5.755ms
  [SPAN] RunnableSequence  latency=3.653ms
  [SPAN] _write  latency=0.001ms
  [SPAN] StateModifier  latency=0ms
  [SPAN] _write  latency=0.001ms
  [SPAN] call_model  latency=1.948ms
  [SPAN] Runnab

### Langfuse Dashboard Screenshot

> Paste a screenshot of the Langfuse dashboard traces page here after running the queries.
>
> Navigate to: https://cloud.langfuse.com → Traces

![Langfuse Traces](../assets/langfuse_traces.png)

## 9. Token Refresh (optional)

If the access token expires (1 hour TTL), use the refresh token.

In [19]:
refresh_resp = cognito.initiate_auth(
    AuthFlow='REFRESH_TOKEN_AUTH',
    AuthParameters={'REFRESH_TOKEN': REFRESH_TOKEN},
    ClientId=COGNITO_CLIENT_ID,
)
JWT_TOKEN = refresh_resp['AuthenticationResult']['AccessToken']
print(f'Token refreshed. New token (first 60 chars): {JWT_TOKEN[:60]}...')

Token refreshed. New token (first 60 chars): eyJraWQiOiJqNVBONE9YYjFuYW1yaWllb3poSkpDTWIrQyt3WkoyZGFCRHlK...
